In [2]:
!pip install -q \
    rank_bm25 \
    sentence-transformers \
    langchain-community \
    langchain-text-splitters \
    pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
import torch
print("torch:", torch.__version__)

torch: 2.10.0+cu128


In [4]:
!pip install -q torchvision --index-url https://download.pytorch.org/whl/cu128

In [5]:
import sentence_transformers, rank_bm25, torch, torchvision
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
CUDA: True


In [3]:
!wget https://digileaders.com/wp-content/uploads/2024/12/2024-State-of-AI-V4.pdf -O state_of_ai.pdf

--2026-04-16 18:48:16--  https://digileaders.com/wp-content/uploads/2024/12/2024-State-of-AI-V4.pdf
Resolving digileaders.com (digileaders.com)... 172.67.177.210, 104.21.91.184, 2606:4700:3033::ac43:b1d2, ...
Connecting to digileaders.com (digileaders.com)|172.67.177.210|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1241033 (1.2M) [application/pdf]
Saving to: ‘state_of_ai.pdf’

state_of_ai.pdf     100%[===================>]   1.18M  1.19MB/s    in 1.0s    

2026-04-16 18:48:18 (1.19 MB/s) - ‘state_of_ai.pdf’ saved [1241033/1241033]



In [4]:
ls -lh

total 1.2M
drwxr-xr-x 1 root root 4.0K Mar 30 13:34 sample_data/
-rw-r--r-- 1 root root 1.2M Mar 31 14:35 state_of_ai.pdf


In [5]:
!rm state_of_ai.pdf
!wget https://aiindex.stanford.edu/wp-content/uploads/2024/05/HAI_AI-Index-Report-2024.pdf -O ai_report.pdf
!ls -lh ai_report.pdf

--2026-04-16 18:49:20--  https://aiindex.stanford.edu/wp-content/uploads/2024/05/HAI_AI-Index-Report-2024.pdf
Resolving aiindex.stanford.edu (aiindex.stanford.edu)... 99.84.215.23, 99.84.215.26, 99.84.215.82, ...
Connecting to aiindex.stanford.edu (aiindex.stanford.edu)|99.84.215.23|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-04-16 18:49:20 ERROR 403: Forbidden.

-rw-r--r-- 1 root root 0 Apr 16 18:49 ai_report.pdf


In [6]:
!rm ai_report.pdf
!wget --user-agent="Mozilla/5.0" https://aiindex.stanford.edu/wp-content/uploads/2024/05/HAI_AI-Index-Report-2024.pdf -O ai_report.pdf
!ls -lh ai_report.pdf

--2026-04-16 18:49:55--  https://aiindex.stanford.edu/wp-content/uploads/2024/05/HAI_AI-Index-Report-2024.pdf
Resolving aiindex.stanford.edu (aiindex.stanford.edu)... 99.84.215.65, 99.84.215.23, 99.84.215.26, ...
Connecting to aiindex.stanford.edu (aiindex.stanford.edu)|99.84.215.65|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-04-16 18:49:55 ERROR 403: Forbidden.

-rw-r--r-- 1 root root 0 Apr 16 18:49 ai_report.pdf


In [7]:
!rm ai_report.pdf
!curl -L -A "Mozilla/5.0" -o ai_report.pdf https://aiindex.stanford.edu/wp-content/uploads/2024/05/HAI_AI-Index-Report-2024.pdf
!ls -lh ai_report.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   919  100   919    0     0  10121      0 --:--:-- --:--:-- --:--:-- 10211
-rw-r--r-- 1 root root 919 Apr 16 18:50 ai_report.pdf


In [6]:
!rm -f ai_report.pdf state_of_ai.pdf
!wget -q https://arxiv.org/pdf/1706.03762 -O attention.pdf
!wget -q https://arxiv.org/pdf/1810.04805 -O bert.pdf
!wget -q https://arxiv.org/pdf/2005.11401 -O rag.pdf
!wget -q https://arxiv.org/pdf/2005.14165 -O gpt3.pdf
!wget -q https://arxiv.org/pdf/2307.09288 -O llama2.pdf
!ls -lh *.pdf

-rw-r--r-- 1 root root 2.2M Apr 12  2024 attention.pdf
-rw-r--r-- 1 root root 757K Jan 22  2023 bert.pdf
-rw-r--r-- 1 root root 6.5M Jan 23  2023 gpt3.pdf
-rw-r--r-- 1 root root  14M Jul 22  2023 llama2.pdf
-rw-r--r-- 1 root root 865K Jan 23  2023 rag.pdf


In [7]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from collections import defaultdict

In [14]:
pdf_files = ["attention.pdf", "bert.pdf", "rag.pdf", "gpt3.pdf", "llama2.pdf"]
all_pages = []
for f in pdf_files:
    pages = PyPDFLoader(f).load()
    all_pages.extend(pages)

full_text = "\n\n".join(p.page_content for p in all_pages)
print(f"Total characters: {len(full_text)}")

Total characters: 670316


In [15]:
def build_chunks(text, chunk_size, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "。", ". ", " "],
    )
    return splitter.create_documents([text])

chunks_small  = build_chunks(full_text, 800)    # ~200 tokens,安全
chunks_medium = build_chunks(full_text, 1600)   # ~400 tokens,甜蜜区
chunks_large  = build_chunks(full_text, 3200)   # ~800 tokens,超 512 截断

print(f"Small:  {len(chunks_small)} chunks")
print(f"Medium:  {len(chunks_medium)} chunks")
print(f"Large: {len(chunks_large)} chunks")

Small:  1141 chunks
Medium:  551 chunks
Large: 312 chunks


In [16]:
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

docs = [c.page_content for c in chunks_medium]
doc_embs = embed_model.encode(docs, show_progress_bar=True, normalize_embeddings=True)

def dense_search(query, top_k=10):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(doc_embs, q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i]) for i in top_idx]

tokenized = [doc.lower().split() for doc in docs]
bm25 = BM25Okapi(tokenized)

def bm25_search(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i]) for i in top_idx]

def rrf_fusion(rankings_lists, k=60, top_n=10):
    from collections import defaultdict
    scores = defaultdict(float)
    for rankings in rankings_lists:
        for rank, (idx, _) in enumerate(rankings):
            scores[idx] += 1 / (k + rank)
    sorted_items = sorted(scores.items(), key=lambda x: -x[1])[:top_n]
    return sorted_items

def hybrid_search(query, top_k=10):
    dense = dense_search(query, top_k=30)
    sparse = bm25_search(query, top_k=30)
    return rrf_fusion([dense, sparse], top_n=top_k)


test_queries = [
    "What is the masked language model objective in BERT?",
    "How does RAG combine retrieval with generation?",
    "Scaled dot-product attention formula",                         # 专有名词 + 关键词
]

for q in test_queries:
    print(f"\n=== Query: {q} ===")
    print("Dense top-3:")
    for i, s in dense_search(q, 3):
        print(f"  [{s:.3f}] {docs[i][:100]}...")
    print("BM25 top-3:")
    for i, s in bm25_search(q, 3):
        print(f"  [{s:.3f}] {docs[i][:100]}...")
    print("Hybrid (RRF) top-3:")
    for i, s in hybrid_search(q, 3):
        print(f"  [{s:.4f}] {docs[i][:100]}...")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/18 [00:00<?, ?it/s]


=== Query: What is the masked language model objective in BERT? ===
Dense top-3:
  [0.884] BERT BERT
E[CLS] E1  E[SEP]... EN E1’ ... EM’
C
 T1
 T[SEP]...
 TN
 T1’ ...
 TM’
[CLS] Tok 1  [SEP]....
  [0.849] 3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditio...
  [0.844] power of the pre-trained representations, espe-
cially for the ﬁne-tuning approaches. The ma-
jor li...
BM25 top-3:
  [19.348] power of the pre-trained representations, espe-
cially for the ﬁne-tuning approaches. The ma-
jor li...
  [18.104] pretraining objective. Our current objective weights every token equally and lacks a notion of what ...
  [16.089] 3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditio...
Hybrid (RRF) top-3:
  [0.0328] power of the pre-trained representations, espe-
cially for the ﬁne-tuning approaches. The ma-
jor li...
  [0.0325] 3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(20

In [17]:
# Cross-encoder 模型（已在 HF 下好预训练版）
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates_idx, top_n=5):
    """candidates_idx: list of doc indices from stage 1"""
    pairs = [(query, docs[i]) for i in candidates_idx]
    scores = cross_encoder.predict(pairs)
    reranked = sorted(zip(candidates_idx, scores), key=lambda x: -x[1])
    return reranked[:top_n]

def full_pipeline(query, retrieve_k=30, final_n=5):
    # Stage 1: Hybrid 召回
    hybrid_results = hybrid_search(query, top_k=retrieve_k)
    candidate_idx = [i for i, _ in hybrid_results]

    # Stage 2: Cross-encoder 精排
    final = rerank(query, candidate_idx, top_n=final_n)
    return final

# ====== 对比：有无 re-ranking ======
for q in test_queries:
    print(f"\n=== Query: {q} ===")

    # 不 rerank：直接用 hybrid top-5
    no_rerank = hybrid_search(q, 5)

    # 先 hybrid 30 再 rerank 到 5
    with_rerank = full_pipeline(q, retrieve_k=30, final_n=5)

    print("Without re-rank (Hybrid top-5):")
    for i, s in no_rerank:
        print(f"  [{s:.4f}] {docs[i][:80]}...")

    print("With re-rank (Hybrid 30 → Cross-enc top-5):")
    for i, s in with_rerank:
        print(f"  [{s:.3f}] {docs[i][:80]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Query: What is the masked language model objective in BERT? ===
Without re-rank (Hybrid top-5):
  [0.0328] power of the pre-trained representations, espe-
cially for the ﬁne-tuning approa...
  [0.0325] 3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we...
  [0.0308] word based only on its context. Unlike left-to-
right language model pre-trainin...
  [0.0302] BERT BERT
E[CLS] E1  E[SEP]... EN E1’ ... EM’
C
 T1
 T[SEP]...
 TN
 T1’ ...
 TM’...
  [0.0295] the vocabulary, as in a standard LM. In all of our
experiments, we mask 15% of a...
With re-rank (Hybrid 30 → Cross-enc top-5):
  [4.398] power of the pre-trained representations, espe-
cially for the ﬁne-tuning approa...
  [4.109] 3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we...
  [4.080] word based only on its context. Unlike left-to-
right language model pre-trainin...
  [2.410] BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding...
  [0

In [18]:
configs = {}
for size, chunks in [(800, chunks_small), (1600, chunks_medium), (3200, chunks_large)]:
    texts = [c.page_content for c in chunks]
    embs = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    configs[size] = {"texts": texts, "embs": embs}

def dense_search_on(cfg, query, top_k=3):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(cfg["embs"], q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i], cfg["texts"][i]) for i in top_idx]

# 对比同一 query 在不同 chunk size 下的召回
for q in test_queries:
    print(f"\n=== Query: {q} ===")
    for size, cfg in configs.items():
        print(f"\n--- chunk_size={size} ---")
        for i, s, t in dense_search_on(cfg, q, 2):
            print(f"  [{s:.3f}] ({len(t)} chars) {t[:120]}...")


=== Query: What is the masked language model objective in BERT? ===

--- chunk_size=800 ---
  [0.899] (416 chars) Encoder Representations from Transformers.
BERT alleviates the previously mentioned unidi-
rectionality constraint by us...
  [0.879] (758 chars) BERT BERT
E[CLS] E1  E[SEP]... EN E1’ ... EM’
C
 T1
 T[SEP]...
 TN
 T1’ ...
 TM’
[CLS] Tok 1  [SEP]... Tok N Tok 1 ... T...

--- chunk_size=1600 ---
  [0.884] (1559 chars) BERT BERT
E[CLS] E1  E[SEP]... EN E1’ ... EM’
C
 T1
 T[SEP]...
 TN
 T1’ ...
 TM’
[CLS] Tok 1  [SEP]... Tok N Tok 1 ... T...
  [0.849] (1580 chars) 3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or...

--- chunk_size=3200 ---
  [0.873] (3194 chars) BERT BERT
E[CLS] E1  E[SEP]... EN E1’ ... EM’
C
 T1
 T[SEP]...
 TN
 T1’ ...
 TM’
[CLS] Tok 1  [SEP]... Tok N Tok 1 ... T...
  [0.837] (3156 chars) BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-We

In [ ]:
class AdvancedRAG:
    def __init__(self, docs, chunk_size=512, overlap=50):
        # Indexing
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=overlap,
            separators=["\n\n", "\n", "。", ". ", " "]
        )
        full = "\n\n".join(docs)
        self.chunks = [c.page_content for c in splitter.create_documents([full])]

        # Dense index
        self.bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.doc_embs = self.bi_encoder.encode(self.chunks, normalize_embeddings=True)

        # Sparse index
        tokenized = [c.lower().split() for c in self.chunks]
        self.bm25 = BM25Okapi(tokenized)

        # Cross-encoder
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def _dense(self, query, k):
        q = self.bi_encoder.encode([query], normalize_embeddings=True)
        scores = np.dot(self.doc_embs, q.T).flatten()
        idx = np.argsort(scores)[::-1][:k]
        return [(i, scores[i]) for i in idx]

    def _sparse(self, query, k):
        scores = self.bm25.get_scores(query.lower().split())
        idx = np.argsort(scores)[::-1][:k]
        return [(i, scores[i]) for i in idx]

    def _rrf(self, rankings_list, k=60, top=20):
        from collections import defaultdict
        scores = defaultdict(float)
        for rankings in rankings_list:
            for rank, (idx, _) in enumerate(rankings):
                scores[idx] += 1 / (k + rank)
        return sorted(scores.items(), key=lambda x: -x[1])[:top]

    def retrieve(self, query, retrieve_k=30, final_n=5, use_hybrid=True, use_rerank=True):
        # Stage 1: 召回
        if use_hybrid:
            dense = self._dense(query, retrieve_k)
            sparse = self._sparse(query, retrieve_k)
            candidates = self._rrf([dense, sparse], top=retrieve_k)
        else:
            candidates = self._dense(query, retrieve_k)

        # Stage 2: 精排
        if use_rerank:
            pairs = [(query, self.chunks[i]) for i, _ in candidates]
            scores = self.cross_encoder.predict(pairs)
            reranked = sorted(zip([i for i, _ in candidates], scores), key=lambda x: -x[1])
            return [(i, s, self.chunks[i]) for i, s in reranked[:final_n]]
        else:
            return [(i, s, self.chunks[i]) for i, s in candidates[:final_n]]

# 用法
rag = AdvancedRAG(docs=[full_text], chunk_size=512)
results = rag.retrieve("What drove revenue growth in 2023?", use_hybrid=True, use_rerank=True)
for i, s, t in results:
    print(f"[{s:.3f}] {t[:150]}...")

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

class AdvacedRAG:
    def __init__(self, docs, chunk_size=512, overlap=50):
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=overlap,
            separator=["\n\n", "\n", "。", ". ", " "]
        )
        full = "\n\n".join(docs)
        self.chunks = [c.page_content for c in splitter.create_document([full])]

        # Dense index
        self.bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.doc_embs = self.bi_encoder.encode(self.chunks, normalize_embeddings=True)

        # Sparse index
        tokenized = [c.lower().split() for c in self.chunks]
        self.bm25 = BM250(tokenized)

        # Cross Encoder
        self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def _dense(self, query, k):
        q = self.bi_coder.encode([query], normalized_embedding=True)
        scores = np.dot(self.doc_embs, q.T).flatten()
        idx = np.argsort(scores)[::-1][:k]
        return [(i, scores[i]) for i in idx]

    def _sparse(self, query, k):
        scores = self.bm25.get_scores(query.lower().split())
        idx = np.argsort(scores)[::-1][:k]
        return [(i, scores[i]) for i in idx]

    def _rrf(self, rankings_list, k=60, top=20):
        from collections import defaultdict
        scores = defaultdict(float)
        for rankings in rankings_list:
            for rank, (idx, _) in enumerate(rankings):
                scores[idx] += 1 /(k+rank)
        return sorted(scores.items(), key=lambda x: -x[1])[:top]

    def retrieve(self, query, retrieve_k=30, final_n=5, use_hybrid=True, use_rerank=True):
        if use_hybrid:
            dense = self._dense(query, retrieve_k)
            sparse = self._sparse(query, retrieve_k)
            candidates = self._rrf([dense, sparse], top=retrieve_k)
        else:
            candidates = self._dense(query, retrieve_k)

        if use_rerank:
            pairs = [(query, self.chunks[i]) for i, _ in candidates]
            scores = self.cross_encoder.predict(pairs)
            reranked = sorted(zip([i for i, _ in candidates], scores), key=lambda x: -x[1])
            return [(i, s, self.chunks[i]) for i, s in reranked[:final_n]]
        else:
            return [(i, s, self.chunks[i]) for i, s in candidates[:final_n]]

rag = AdvancedRAG(docs=[full_text], chunk_size=512)
results = rag.retrieve("What drove revenue growth in 2023?", use_hybrid=True, use_rerank=True)
for i, s, t in results:
    print(f"[{s:.3f}] {t[:150]}...")


ModuleNotFoundError: No module named 'langchain_text_splitters'